In [2]:
# ============================================================
# CÉLULA 1 — INSTALAÇÃO DE DEPENDÊNCIAS
# ============================================================

# Instala o Selenium e bibliotecas de apoio.
!pip install -q selenium pandas beautifulsoup4

# Remove qualquer chromedriver quebrado do apt.
!apt-get remove -y chromium-chromedriver > /dev/null 2>&1

# Atualiza a lista de pacotes.
!apt-get update -qq

# Instala utilitários para baixar e descompactar arquivos.
!apt-get install -y -qq wget unzip curl > /dev/null 2>&1

# Baixa e instala o Google Chrome Stable.
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb > /dev/null 2>&1
!apt-get install -f -y -qq > /dev/null 2>&1

# Mostra a versão do Chrome instalada.
!google-chrome --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Google Chrome 156.0.8078.47 


In [14]:
# ============================================================
# BIBLIOMETRIA DOS ANAIS DO SBSR (INPE) COM AUTORES
# ============================================================

import re
import time
import subprocess
import unicodedata
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ------------------------------------------------------------
# 1. CONFIGURAR CHROMEDRIVER
# ------------------------------------------------------------
versao_chrome = subprocess.check_output(["google-chrome", "--version"]).decode()
versao_chrome = re.search(r"(\d+\.\d+\.\d+\.\d+)", versao_chrome).group(1)
print(f"Chrome instalado: {versao_chrome}")

url_driver = (
    f"https://storage.googleapis.com/chrome-for-testing-public/"
    f"{versao_chrome}/linux64/chromedriver-linux64.zip"
)
!wget -q {url_driver} -O /tmp/chromedriver.zip
!unzip -o /tmp/chromedriver.zip -d /tmp/ > /dev/null
!chmod +x /tmp/chromedriver-linux64/chromedriver
!mv /tmp/chromedriver-linux64/chromedriver /usr/local/bin/chromedriver
!chromedriver --version

# ------------------------------------------------------------
# 2. URLS DE SUMÁRIO POR EDIÇÃO
# ------------------------------------------------------------
SUMARIOS = {
    1978: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/09.25.20.31/doc/@sumario.htm",
    1982: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/10.01.23.18/doc/@sumario.htm",
    1984: "http://marte.sid.inpe.br/col/ltid.inpe.br/sbsr/2009/04.03.17.43/doc/@sumario.htm",
    1986: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2009/03.03.11.29/doc/@sumario.htm",
    1988: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/07.28.17.53/doc/@sumario.htm",
    1990: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/08.26.21.29/doc/@sumario.htm",
    1993: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/06.23.20.41/doc/@sumario.htm",
    1996: "http://marte.sid.inpe.br/col/dpi.inpe.br/lise/2004/08.25.13.20/doc/@sumario.htm",
    1998: "http://marte.sid.inpe.br/col/dpi.inpe.br/lise/2004/08.11.17.13/doc/@sumario.htm",
    2001: "http://marte.sid.inpe.br/col/dpi.inpe.br/lise/2001/12.28.15.53/doc/@sumario.htm",
    2003: "http://marte.sid.inpe.br/col/ltid.inpe.br/sbsr/2003/03.28.12.35/doc/@sumario.htm",
    2005: "http://marte.sid.inpe.br/col/ltid.inpe.br/sbsr/2005/02.12.16.31/doc/@sumario.htm",
    2007: "http://marte.sid.inpe.br/col/dpi.inpe.br/sbsr@80/2007/01.31.19.10/doc/@sumario.htm",
    2009: "http://marte.sid.inpe.br/col/dpi.inpe.br/sbsr@80/2009/03.02.18.19/doc/@sumario.htm",
    2011: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte/2011/09.28.16.42/doc/@sumario.htm",
    2013: "http://marte2.sid.inpe.br/col/dpi.inpe.br/marte2/2013/05.28.22.25/doc/@sumario.htm",
    2015: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2015/05.31.21.54/doc/@sumario.htm",
    2017: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2017/09.25.14.55/doc/@sumario.htm",
    2019: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2019/11.08.12.52/doc/@sumario.htm",
    2023: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2023/05.18.17.15/doc/@sumario.htm",
    2025: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2025/09.17.19.06/doc/@sumario.htm",
}

ROMANOS = {1978: "I", 1982: "II", 1984: "III", 1986: "IV", 1988: "V",
           1990: "VI", 1993: "VII", 1996: "VIII", 1998: "IX", 2001: "X",
           2003: "XI", 2005: "XII", 2007: "XIII", 2009: "XIV", 2011: "XV",
           2013: "XVI", 2015: "XVII", 2017: "XVIII", 2019: "XIX",
           2023: "XX", 2025: "XXI"}

# ------------------------------------------------------------
# 3. PALAVRAS-CHAVE POR CATEGORIA (originais, sem alteração)
# ------------------------------------------------------------
CATEGORIAS = {
    "Deslizamentos": ["deslizamento", "escorregamento", "desmoronamento", "movimento de massa", "movimentos de massa", "landslide"],
    "Inundação": ["inundação", "inundações", "enchente", "alagamento", "enxurrada", "flood", "flooding"],
    "Secas": ["seca", "estiagem", "estiagens", "desertificação", "drought", "desertification"]
}

# ------------------------------------------------------------
# 4. FUNÇÕES AUXILIARES
# ------------------------------------------------------------
def normalizar(texto):
    """
    Normaliza para comparação bibliométrica:
      - minúsculas
      - sem acentos
      - hífens, underscores e pontuação → espaço
      - espaços colapsados
    """
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[\-_/\\.,;:!?()\[\]{}'\"`´]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def construir_padroes():
    """
    Compila uma regex por termo, cobrindo:
      - singular e plural regular (s / es)
      - fronteira de palavra (evita 'seca' casar 'secagem')
    Retorna lista de (categoria, termo_original, regex).
    """
    padroes = []
    for cat, termos in CATEGORIAS.items():
        for termo in termos:
            base = normalizar(termo)
            padrao = re.compile(
                rf"\b{re.escape(base)}(s|es)?\b",
                flags=re.IGNORECASE,
            )
            padroes.append((cat, termo, padrao))
    return padroes

PADROES = construir_padroes()

PAGINAS_IGNORAR = {
    "capa.htm", "pagina_rosto.htm", "ficha_catalografica.htm",
    "prefacio.htm", "comissao.htm", "@indice.htm", "indice.htm",
    "@sumario.htm", "sumario.htm", "pagina_rosto.html",
}

def eh_linha_institucional(linha):
    linha_lower = linha.lower()

    if any(p in linha_lower for p in [
        "gerado automaticamente", "urlibservice", "sumario", "sumário",
        "pagina de rosto", "página de rosto", "dados catalograficos",
        "dados catalográficos", "prefacio", "prefácio",
        "comissoes", "comissões", "indice por autor", "índice por autor",
        "exemplo para referenciar", "link para os temas",
        "simpósio brasileiro", "simposio brasileiro", "sbsr",
        "urlib", "your browser cannot display",
    ]):
        return True

    if re.match(r"^\d{1,2}\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]", linha):
        return True

    if len(linha) < 15:
        return True

    return False

def extrair_artigos_por_links(html, base_url):
    """
    Extrai (título, autores, link) a partir de tags <a>.
    Separa título de autores quando ambos estão dentro do mesmo <a>:
      - por quebras de linha reais (get_text com '\\n')
      - por regex de nomes próprios
    """
    soup = BeautifulSoup(html, "html.parser")
    resultados = []

    padrao_autores = re.compile(
        r"([A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+"
        r"(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+)+"
        r"(?:\s*;\s*[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+"
        r"(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+)+)*)"
    )

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()

        # --- SEPARAR TÍTULO DE AUTORES ---
        texto_nl = a.get_text("\n", strip=True)
        linhas = [l.strip() for l in texto_nl.split("\n") if l.strip()]

        if len(linhas) >= 2:
            titulo = linhas[0]
            autores = "; ".join(linhas[1:])
        else:
            texto_plano = a.get_text(" ", strip=True)
            m = padrao_autores.search(texto_plano)
            if m and m.start() > 25:
                titulo = texto_plano[:m.start()].strip()
                autores = m.group(1).strip()
            else:
                titulo = texto_plano
                autores = ""

        # --- FILTROS ---
        if href.startswith("#"):
            continue
        if href in PAGINAS_IGNORAR:
            continue
        if any(p in href.lower() for p in
               ["capa", "rosto", "ficha", "prefacio", "comissao",
                "indice", "sumario", "topo"]):
            continue
        if not re.search(r"\.(pdf|htm|html)$", href, re.IGNORECASE):
            continue
        if len(titulo) < 25:
            continue
        if re.fullmatch(r"[\d\s\.\-]+", titulo):
            continue

        link_absoluto = urljoin(base_url, href)
        resultados.append({
            "titulo": titulo,
            "autores": autores,
            "link": link_absoluto,
        })

    vistos = set()
    unicos = []
    for r in resultados:
        if r["titulo"] not in vistos:
            vistos.add(r["titulo"])
            unicos.append(r)
    return unicos

def extrair_artigos_por_texto(html):
    """
    Extrai (título, autores) do texto puro.
    Trata dois casos:
      A) título e autores grudados na mesma linha (com ';')
      B) título em uma linha, autores na linha seguinte
    """
    soup = BeautifulSoup(html, "html.parser")
    texto = soup.get_text("\n", strip=True)
    linhas = [l.strip() for l in texto.split("\n") if l.strip()]

    artigos = []
    i = 0
    while i < len(linhas):
        linha_atual = linhas[i]
        linha_seguinte = linhas[i + 1] if i + 1 < len(linhas) else ""

        if eh_linha_institucional(linha_atual):
            i += 1
            continue

        # CASO A: título e autores na mesma linha
        if (";" in linha_atual
            and len(linha_atual) > 60
            and not re.match(r"^\d{1,2}\s", linha_atual)):

            partes = linha_atual.split(";")
            titulo_cand = partes[0].strip()
            autores_cand = "; ".join(p.strip() for p in partes[1:])

            if len(titulo_cand) > 25:
                artigos.append({
                    "titulo": titulo_cand,
                    "autores": autores_cand,
                    "link": "",
                })
                i += 1
                continue

        # CASO B: título na linha atual, autores na próxima
        if (len(linha_atual) > 25
            and ";" not in linha_atual
            and not re.match(r"^\d{1,2}\s", linha_atual)
            and not linha_atual.endswith(".")):

            if (";" in linha_seguinte
                or re.search(r"[A-Z][a-z]+,\s*[A-Z]\.", linha_seguinte)):

                artigos.append({
                    "titulo": linha_atual,
                    "autores": linha_seguinte,
                    "link": "",
                })
                i += 2
                continue

        i += 1

    return artigos

def extrair_autores_da_pagina_artigo(html):
    """
    Extrai autores de uma página de artigo (fallback para edições
    antigas onde o sumário não traz autores).
    """
    soup = BeautifulSoup(html, "html.parser")
    autores = []

    metas = soup.find_all("meta", {"name": "citation_author"})
    for m in metas:
        if m.get("content"):
            autores.append(m["content"].strip())

    if not autores:
        texto = soup.get_text(" ", strip=True)
        padroes = [
            r'Autor(?:es)?\s*[:\-]\s*([A-Z][^.]{5,150})',
            r'By\s+([A-Z][^.]{5,150})',
        ]
        for p in padroes:
            matches = re.findall(p, texto)
            for m in matches:
                nomes = re.split(r'[;,]', m)
                for n in nomes:
                    n = n.strip()
                    if 5 < len(n) < 80:
                        autores.append(n)
            if autores:
                break

    vistos = set()
    limpos = []
    for a in autores:
        a = a.strip()
        if a and a not in vistos and len(a) > 3:
            vistos.add(a)
            limpos.append(a)
    return "; ".join(limpos) if limpos else ""

def extrair_artigos_com_frames(driver, url):
    """
    Acessa a URL, detecta frames e extrai artigos.
    Tenta primeiro o método por links; se falhar, usa o método por texto.
    """
    try:
        driver.set_page_load_timeout(40)
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(3)

        artigos = extrair_artigos_por_links(driver.page_source, url)

        if len(artigos) < 5:
            candidatos = extrair_artigos_por_texto(driver.page_source)
            if len(candidatos) > len(artigos):
                artigos = candidatos

        if len(artigos) < 5:
            frames = (driver.find_elements(By.TAG_NAME, "frame") +
                      driver.find_elements(By.TAG_NAME, "iframe"))
            for frame in frames:
                try:
                    driver.switch_to.frame(frame)
                    time.sleep(2)
                    cand_links = extrair_artigos_por_links(
                        driver.page_source, driver.current_url
                    )
                    cand_texto = extrair_artigos_por_texto(driver.page_source)
                    driver.switch_to.default_content()

                    melhor = max([artigos, cand_links, cand_texto], key=len)
                    if len(melhor) > len(artigos):
                        artigos = melhor
                except Exception:
                    driver.switch_to.default_content()

        return artigos

    except Exception as e:
        print(f"      Erro: {type(e).__name__}: {str(e)[:100]}")
        try:
            driver.switch_to.default_content()
        except Exception:
            pass
        return []

# ------------------------------------------------------------
# 5. CONFIGURAR SELENIUM
# ------------------------------------------------------------
options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--lang=pt-BR")
options.add_argument(
    "user-agent=Mozilla/5.0 (X11; Linux x86_64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

service = Service("/usr/local/bin/chromedriver")
driver = webdriver.Chrome(service=service, options=options)

# ------------------------------------------------------------
# 6. LAÇO PRINCIPAL
# ------------------------------------------------------------
ocorrencias = []
falhas = []

for ano, url_sumario in SUMARIOS.items():
    print(f"\n=== Edição de {ano} ===")
    print(f"   Sumário: {url_sumario}")

    artigos = extrair_artigos_com_frames(driver, url_sumario)

    if not artigos:
        print(f"   Nenhum título encontrado para {ano}.")
        falhas.append(ano)
        continue

    print(f"   {len(artigos)} artigos encontrados.")

    # Deduplicação bibliométrica: 1 artigo = 1 registro por edição
    vistos_na_edicao = set()

    for idx, item in enumerate(artigos):
        titulo = item["titulo"]
        autores = item.get("autores", "")
        link = item.get("link", "")

        # Rede de segurança: se autores vieram vazios mas o título tem ';',
        # separa título de autores na hora (fallback para casos residuais).
        if not autores and ";" in titulo:
            partes = titulo.split(";")
            if len(partes[0].strip()) > 25:
                autores = "; ".join(p.strip() for p in partes[1:])
                titulo = partes[0].strip()

        t_norm = normalizar(titulo)

        if t_norm in vistos_na_edicao:
            continue
        vistos_na_edicao.add(t_norm)

        # Matching com regex (fronteira de palavra + plural)
        matches = []
        for cat, termo_original, regex in PADROES:
            m = regex.search(t_norm)
            if m:
                matches.append((cat, termo_original, m.group(0)))

        if not matches:
            continue

        # Busca autores somente se necessário
        if not autores and link:
            try:
                driver.set_page_load_timeout(20)
                driver.get(link)
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.TAG_NAME, "body"))
                )
                time.sleep(1)
                autores = extrair_autores_da_pagina_artigo(driver.page_source)
            except Exception as e:
                print(f"      Erro ao buscar autores de "
                      f"'{titulo[:50]}...': {type(e).__name__}")

        # Agrupa por categoria: 1 linha por (artigo × categoria)
        por_categoria = {}
        for cat, termo, encontrado in matches:
            por_categoria.setdefault(cat, []).append((termo, encontrado))

        for cat, pares in por_categoria.items():
            palavras = "; ".join(sorted({t for t, _ in pares}))
            evidencias = "; ".join(sorted({e for _, e in pares}))
            ocorrencias.append({
                "Categoria": cat,
                "Palavras-Chave": palavras,
                "Evidencia_Titulo": evidencias,
                "Edicao SBSR": ROMANOS.get(ano, str(ano)),
                "Ano": ano,
                "Titulo": titulo,
                "Autores": autores,
            })

        if (idx + 1) % 100 == 0:
            print(f"      ... {idx + 1}/{len(artigos)} artigos processados")

driver.quit()

# ------------------------------------------------------------
# 7. SALVAR CSV
# ------------------------------------------------------------
df = pd.DataFrame(ocorrencias, columns=[
    "Categoria", "Palavras-Chave", "Evidencia_Titulo",
    "Edicao SBSR", "Ano", "Titulo", "Autores"
])
df = df.sort_values(["Ano", "Categoria", "Titulo"]).reset_index(drop=True)

caminho = "/content/bibliometria_sbsr_com_autores.csv"
df.to_csv(caminho, index=False, encoding="utf-8-sig", sep=";")

# ------------------------------------------------------------
# 8. RESUMOS BIBLIOMÉTRICOS
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(f"Total de registros (artigo × categoria): {len(df)}")
print(f"CSV salvo em: {caminho}")
print("=" * 60)

if not df.empty:
    print("\n--- Artigos por categoria ---")
    print(df["Categoria"].value_counts().to_string())

    print("\n--- Artigos por edição ---")
    # Rótulo acentuado apenas na exibição; a coluna continua "Edicao SBSR"
    print(
        df.groupby(["Ano", "Edicao SBSR"])
          .size()
          .rename_axis(["Ano", "Edição do SBSR"])
          .to_string()
    )

    print("\n--- Top 10 autores (contagem de aparições) ---")
    autores_serie = (
        df["Autores"].dropna().str.split(";").explode().str.strip()
    )
    autores_serie = autores_serie[autores_serie != ""]
    print(autores_serie.value_counts().head(10).to_string())

    print("\n--- Primeiras 20 linhas ---")
    print(df.head(20).to_string(index=False))

    # --------------------------------------------------------
    # Diagnóstico: verificar se ainda há títulos com ';'
    # (sinal de autores grudados no título)
    # --------------------------------------------------------
    suspeitos = df[df["Titulo"].str.contains(";", na=False)]
    print(f"\n--- Títulos suspeitos (com ';'): {len(suspeitos)} ---")
    if not suspeitos.empty:
        print(suspeitos[["Titulo", "Autores"]].head(10).to_string(index=False))

    sem_autores = df[df["Autores"].fillna("").str.strip() == ""]
    print(f"\n--- Registros sem autores: {len(sem_autores)} "
          f"({100 * len(sem_autores) / max(len(df), 1):.1f}%) ---")
else:
    print("Nenhuma ocorrência encontrada.")

if falhas:
    print(f"\nEdições sem dados: {falhas}")

Chrome instalado: 156.0.8078.47
ChromeDriver 156.0.8078.47 (1d17a2e2ee147bb7d6ae2ab6abdb3a2510c17775-refs/branch-heads/8078@{#1313})

=== Edição de 1978 ===
   Sumário: http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/09.25.20.31/doc/@sumario.htm
   84 artigos encontrados.

=== Edição de 1982 ===
   Sumário: http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/10.01.23.18/doc/@sumario.htm
   107 artigos encontrados.

=== Edição de 1984 ===
   Sumário: http://marte.sid.inpe.br/col/ltid.inpe.br/sbsr/2009/04.03.17.43/doc/@sumario.htm
   47 artigos encontrados.

=== Edição de 1986 ===
   Sumário: http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2009/03.03.11.29/doc/@sumario.htm
   211 artigos encontrados.

=== Edição de 1988 ===
   Sumário: http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/07.28.17.53/doc/@sumario.htm
   103 artigos encontrados.

=== Edição de 1990 ===
   Sumário: http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/08.26.21.29/doc/@sumario.htm
   133 artig